In [0]:
# Cell 1: Configuration
import random
import time
import requests
import numpy as np
from datetime import datetime, timedelta
from pyspark.sql import functions as F

ENDPOINT_NAME = "gdpr-agent-staging"
QUESTION_POOL_TABLE = "main.default.gdpr_agent_question_pool"

print(f"✅ Configuration loaded")

In [0]:
# Cell 2: Sample Questions from Pool

# Determine number of queries (normal distribution)
num_queries = int(np.random.normal(30, 10))
num_queries = max(1, min(num_queries, 60))

print(f"🎲 Randomly selected: {num_queries} queries")

# Random sample from question pool (with replacement for variety)
sampled_questions = spark.table(QUESTION_POOL_TABLE) \
    .orderBy(F.rand()) \
    .limit(num_queries) \
    .toPandas()

print(f"✅ Sampled {len(sampled_questions)} questions from pool")

In [0]:
# Cell 3: Send Queries with LLM Rewriting

import sys
sys.path.insert(0, '/Workspace/Repos/vernonc.lam@gmail.com/GDPR-agent')

from monitoring.utils.request_logger import RequestLogger
from openai import OpenAI
from databricks.sdk import WorkspaceClient

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
endpoint_url = f"https://{workspace_url}/serving-endpoints/{ENDPOINT_NAME}/invocations"

# Initialize logger
logger = RequestLogger("main.default.gdpr_agent_inference_logs")

# Initialize OpenAI client for query rewriting
w = WorkspaceClient()
openai_key = w.dbutils.secrets.get(scope="openai", key="GDPR_agent")
openai_client = OpenAI(api_key=openai_key)

def rewrite_question(original_question: str) -> str:
    """
    Use LLM to rewrite the question with natural variations.
    Keeps the same intent but changes phrasing, formality, or detail level.
    """
    prompt = f"""Rewrite this GDPR-related question with natural variation. Keep the same intent and topic, but change the phrasing, formality level, or add/remove minor details. Make it sound like a different person asking the same thing.

Original: {original_question}

Rewritten (just output the question, nothing else):"""
    
    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.9,  # High temperature for variety
            max_tokens=150
        )
        rewritten = response.choices[0].message.content.strip()
        return rewritten
    except Exception as e:
        print(f"   ⚠️  Rewrite failed: {e}, using original")
        return original_question

# Prepare query schedule
query_schedule = []
for _, row in sampled_questions.iterrows():
    query_schedule.append({
        'scheduled_time': datetime.now(),
        'original_question': row['question'],
        'question_id': row['question_id'],
        'category': row['category']
    })

print(f"📅 Scheduled {len(query_schedule)} queries\n")

# Execute queries
results = []

for i, item in enumerate(query_schedule, 1):
    print(f"📤 [{i}/{len(query_schedule)}] {item['category']}")
    print(f"   Original: {item['original_question'][:60]}...")
    
    # Rewrite the question with LLM
    rewritten_question = rewrite_question(item['original_question'])
    print(f"   Rewritten: {rewritten_question[:60]}...")
    
    payload = {"dataframe_split": {"columns": ["question"], "data": [[rewritten_question]]}}
    
    try:
        start_time = datetime.now()
        response = requests.post(
            endpoint_url, 
            headers={
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/json"
            },
            json=payload, 
            timeout=180  # Increased timeout for cold starts
        )
        end_time = datetime.now()
        latency_ms = (end_time - start_time).total_seconds() * 1000
        
        if response.status_code == 200:
            result = response.json()
            pred = result['predictions'][0]
            
            answer = pred.get('answer', '')
            context = str(pred.get('context', []))
            metadata = pred.get('metadata', {})
            request_id = metadata.get('request_id', item['question_id'])
            
            print(f"   ✅ Success ({latency_ms/1000:.2f}s)")
            
            # Log success (log the rewritten question that was actually sent)
            logger.log_success(
                request_id=request_id,
                question=rewritten_question,
                answer=answer,
                context=context,
                latency_ms=latency_ms,
                timestamp=start_time
            )
            
            results.append({'status': 'success', 'latency': latency_ms/1000})
            
        else:
            print(f"   ❌ Error: {response.status_code}")
            
            # Log error
            logger.log_error(
                request_id=item['question_id'],
                question=rewritten_question,
                error_message=f"HTTP {response.status_code}: {response.text[:200]}",
                latency_ms=latency_ms,
                timestamp=start_time
            )
            
            results.append({'status': 'error'})
            
    except Exception as e:
        print(f"   ❌ Exception: {e}")
        
        # Log exception
        logger.log_exception(
            request_id=item['question_id'],
            question=rewritten_question if 'rewritten_question' in locals() else item['original_question'],
            exception=e,
            timestamp=datetime.now()
        )
        
        results.append({'status': 'exception'})
    
    # Small delay between requests
    time.sleep(2)

# Write all logs to Delta
print(f"\n💾 Flushing logs to Delta...")
logger.flush()

print(f"\n✅ Complete: {sum(1 for r in results if r['status']=='success')}/{len(results)} successful")